In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Conv2D,MaxPooling2D,BatchNormalization,Flatten,Dense,Dropout)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam,SGD,RMSprop
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Sequential
train_path="/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train"
test_path="/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test"
val_path="/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/val"
train_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)
train_data=train_datagen.flow_from_directory(train_path,target_size=(128,128),batch_size=32,class_mode='binary')
test_data=test_datagen.flow_from_directory(test_path,target_size=(128,128),batch_size=32,class_mode='binary')
val_data=test_datagen.flow_from_directory(val_path,target_size=(128,128),batch_size=32,class_mode='binary')
def create_model(optimizer_name):
    model=Sequential()
    model.add(Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Conv2D(64,(3,3),activation='relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Flatten())
    model.add(Dense(128,activation='relu',kernel_regularizer='l2'))
    model.add(Dropout(0.5))
    model.add(Dense(1,activation='sigmoid'))
    if optimizer_name == 'adam':
        optimizer = Adam()
    elif optimizer_name == 'sgd':
        optimizer = SGD()
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop()
    model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
    return model
adam_model = create_model('adam')
history_adam = adam_model.fit(train_data,validation_data=val_data,epochs=5)
sgd_model=create_model('sgd')
sgd_history=sgd_model.fit(train_data,validation_data=val_data,epochs=5)
rms_model = create_model('rmsprop')
history_rms = rms_model.fit(train_data,validation_data=val_data,epochs=5)
adam_loss, adam_acc = adam_model.evaluate(test_data)
sgd_loss, sgd_acc = sgd_model.evaluate(test_data)
rms_loss, rms_acc = rms_model.evaluate(test_data)
print("Adam Accuracy :", adam_acc)
print("SGD Accuracy :", sgd_acc)
print("RMSprop Accuracy :", rms_acc)